# Financial Fraud Detection

## Phase 13 - Real-Time Fraud Prediction

This notebook:

- Connects Spark Streaming to Kafka
- Reads live transactions
- Parses JSON
- Loads the trained Random Forest model
- Predicts fraud in real time

## Imports

In [ ]:
import json
import joblib
import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    from_json,
    when,
    log1p
)

from pyspark.sql.types import *

## Layout Cleaning

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

## Create Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("RealTimeFraudPrediction")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1"
    )
    .getOrCreate()
)

## Stop ALL running queries

In [ ]:
for q in spark.streams.active:
    print("Stopping:", q.id)
    q.stop()

In [ ]:
spark.streams.active

## Load Best Model

In [ ]:
model = joblib.load("../models/best_model.pkl")

print(model)

## Load Metadata

In [ ]:
with open("../models/best_model_metadata.json","r") as f:
    metadata = json.load(f)

metadata

## Read Kafka Stream

In [ ]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","localhost:9092")
    .option("subscribe","fraud-transactions")
    .load()
)

## Schema

In [ ]:
schema = StructType([

    StructField("Time", DoubleType()),

    StructField("V1", DoubleType()),
    StructField("V2", DoubleType()),
    StructField("V3", DoubleType()),
    StructField("V4", DoubleType()),
    StructField("V5", DoubleType()),
    StructField("V6", DoubleType()),
    StructField("V7", DoubleType()),
    StructField("V8", DoubleType()),
    StructField("V9", DoubleType()),
    StructField("V10", DoubleType()),
    StructField("V11", DoubleType()),
    StructField("V12", DoubleType()),
    StructField("V13", DoubleType()),
    StructField("V14", DoubleType()),
    StructField("V15", DoubleType()),
    StructField("V16", DoubleType()),
    StructField("V17", DoubleType()),
    StructField("V18", DoubleType()),
    StructField("V19", DoubleType()),
    StructField("V20", DoubleType()),
    StructField("V21", DoubleType()),
    StructField("V22", DoubleType()),
    StructField("V23", DoubleType()),
    StructField("V24", DoubleType()),
    StructField("V25", DoubleType()),
    StructField("V26", DoubleType()),
    StructField("V27", DoubleType()),
    StructField("V28", DoubleType()),

    StructField("Amount", DoubleType()),
    StructField("transaction_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("Class", DoubleType())
])

## Parse JSON

In [ ]:
parsed_df = (
    stream_df
    .selectExpr("CAST(value AS STRING)")
    .select(
        from_json(
            col("value"),
            schema
        ).alias("data")
    )
    .select("data.*")
)

## Verify Schema

In [ ]:
parsed_df.printSchema()

## Feature Engineering

In [ ]:
feature_df = (

    parsed_df

    .withColumn(
        "Large_Transaction",
        when(col("Amount") > 200, 1).otherwise(0)
    )

    .withColumn(
        "Log_Amount",
        log1p(col("Amount"))
    )

    .withColumn(
        "Amount_Level",
        when(col("Amount") < 10, 0)
        .when(col("Amount") < 100, 1)
        .when(col("Amount") < 500, 2)
        .otherwise(3)
    )

)

## Select Features in Model Order

In [ ]:
feature_columns = list(model.feature_names_in_)

prediction_df = feature_df.select(

    "transaction_id",

    "event_timestamp",

    *feature_columns

)

## Verify Columns

In [ ]:
prediction_df.printSchema()

## Create Prediction Function

In [ ]:
def predict_batch(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    # Preserve metadata
    metadata_df = pdf[
        [
            "transaction_id",
            "event_timestamp"
        ]
    ].copy()

    # Features for prediction
    X = pdf[model.feature_names_in_]

    # Predictions
    predictions = model.predict(X)

    probabilities = model.predict_proba(X)[:, 1]

    # Final output
    result = metadata_df.copy()

    result["Amount"] = pdf["Amount"]

    result["Prediction"] = predictions

    result["Status"] = result["Prediction"].map({
        0: "✅ Genuine",
        1: "🚨 Fraud"
    })

    result["Fraud Probability"] = probabilities.round(6)

    result["Risk Level"] = pd.cut(
        result["Fraud Probability"],
        bins=[-0.01, 0.20, 0.50, 0.80, 1.00],
        labels=[
            "🟢 Low",
            "🟡 Medium",
            "🟠 High",
            "🔴 Critical"
        ]
    )

    print("\n")
    print("=" * 120)
    print(f"Batch {batch_id}")
    print("=" * 120)

    print(
        result[
            [
                "transaction_id",
                "Amount",
                "Fraud Probability",
                "Risk Level",
                "Status"
            ]
        ]
    )

    frauds = result[result["Prediction"] == 1]

    if not frauds.empty:

        print("\n")
        print("🚨" * 25)
        print("🚨 LIVE FRAUD ALERT 🚨")
        print("🚨" * 25)

        print(
            frauds[
                [
                    "transaction_id",
                    "Amount",
                    "Fraud Probability",
                    "Risk Level"
                ]
            ]
        )

    print("\n")
    print(f"Processed {len(result)} transaction(s)")

## Start Streaming Prediction

In [ ]:
query = (

    prediction_df.writeStream

    .foreachBatch(predict_batch)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../artifacts/checkpoints/realtime_prediction"
    )

    .start()

)

## Streaming Status

In [ ]:
print(query.status)

## Active Streams

In [ ]:
spark.streams.active

## Keep Alive

In [ ]:
query.awaitTermination()

## Run to End the stream

In [ ]:
query.stop()

print("Streaming stopped.")

In [ ]:
spark.streams.active